In [ ]:
import requests
import pandas as pd
from tqdm import tqdm
import time

## Configuración principal

Reemplazar el ID institucional por la institución que se quiere analizar.

Ejemplo:

MIT → https://openalex.org/I63966007

PUCV → https://openalex.org/institutions/i130474213

USM → https://openalex.org/institutions/I75778554

USM 2 → https://openalex.org/institutions/I4210138643

UNAB → https://openalex.org/institutions/i13897259

USACH → https://openalex.org/institutions/I10457146

UV → https://openalex.org/institutions/I79274474

UDEConce → https://openalex.org/institutions/I172787465

UCHILE → https://openalex.org/institutions/I69737025

PUC → https://openalex.org/institutions/I162148367

In [ ]:
# =========================
# CONFIGURACIÓN
# =========================

INSTITUTION_ID = "I162148367"   # Reemplazar
START_YEAR = 2019
END_YEAR = 2026

# Engineering & Technology en OpenAlex
# Corresponde al dominio principal:
# https://openalex.org/domains/3

ENGINEERING_DOMAIN_ID = 3

PER_PAGE = 200

In [ ]:
# Función para obtener los papers desde OpenAlex
def get_engineering_works(institution_id, start_year, end_year):

    all_works = []
    cursor = "*"

    while cursor:
        url = "https://api.openalex.org/works"

        params = {
            "filter": (
                f"institutions.id:https://openalex.org/{institution_id},"
                f"from_publication_date:{start_year}-01-01,"
                f"to_publication_date:{end_year}-12-31,"
                f"primary_topic.domain.id:{ENGINEERING_DOMAIN_ID}"
            ),
            "per-page": PER_PAGE,
            "cursor": cursor,
            "select": "id,display_name,publication_year,cited_by_count,primary_topic"
        }

        response = requests.get(url, params=params)

        if response.status_code != 200:
            print("Error:", response.status_code)
            print(response.text)
            break

        data = response.json()

        results = data.get("results", [])

        if not results:
            break

        all_works.extend(results)

        cursor = data["meta"].get("next_cursor")

        print(f"Papers descargados: {len(all_works)}")

        time.sleep(0.1)

    return all_works

In [ ]:
# Ejecutar la función principal

works = get_engineering_works(
    INSTITUTION_ID,
    START_YEAR,
    END_YEAR
)

print(f"Total papers descargados: {len(works)}")

Papers descargados: 200
Papers descargados: 400
Papers descargados: 600
Papers descargados: 800
Papers descargados: 1000
Papers descargados: 1200
Papers descargados: 1400
Papers descargados: 1600
Papers descargados: 1800
Papers descargados: 2000
Papers descargados: 2200
Papers descargados: 2400
Papers descargados: 2600
Papers descargados: 2800
Papers descargados: 3000
Papers descargados: 3200
Papers descargados: 3400
Papers descargados: 3600
Papers descargados: 3800
Papers descargados: 4000
Papers descargados: 4200
Papers descargados: 4400
Papers descargados: 4600
Papers descargados: 4800
Papers descargados: 5000
Papers descargados: 5200
Papers descargados: 5400
Papers descargados: 5600
Papers descargados: 5800
Papers descargados: 6000
Papers descargados: 6200
Papers descargados: 6400
Papers descargados: 6600
Papers descargados: 6800
Papers descargados: 7000
Papers descargados: 7200
Papers descargados: 7400
Papers descargados: 7600
Papers descargados: 7800
Papers descargados: 8000
Pape

In [ ]:
# Acoplar a un DataFrame

rows = []

for w in works:

    rows.append({
        "paper_id": w.get("id"),
        "title": w.get("display_name"),
        "year": w.get("publication_year"),
        "citations": w.get("cited_by_count", 0),
        "domain": w.get("primary_topic", {})
    })


df = pd.DataFrame(rows)

print(df.head())

                           paper_id  \
0  https://openalex.org/W4225246927   
1  https://openalex.org/W4280589633   
2  https://openalex.org/W2888259247   
3  https://openalex.org/W2999414134   
4  https://openalex.org/W2924886534   

                                               title  year  citations  \
0                Brain charts for the human lifespan  2022       1726   
1  First Sagittarius A* Event Horizon Telescope R...  2022       1719   
2  The Simons Observatory: science goals and fore...  2019       1529   
3  Principles for knowledge co-production in sust...  2020       1511   
4  Amphibian fungal panzootic causes catastrophic...  2019       1352   

                                              domain  
0  {'id': 'https://openalex.org/T14393', 'display...  
1  {'id': 'https://openalex.org/T10744', 'display...  
2  {'id': 'https://openalex.org/T10818', 'display...  
3  {'id': 'https://openalex.org/T10119', 'display...  
4  {'id': 'https://openalex.org/T10332', 'display..

In [ ]:
# Definición de las métricas principales

TOTAL_PAPERS = len(df)
TOTAL_CITATIONS = df["citations"].sum()
AVG_CITATIONS = df["citations"].mean()

print("========================")
print("RESULTADOS")
print("========================")
print(f"Total papers: {TOTAL_PAPERS}")
print(f"Total citations: {TOTAL_CITATIONS}")
print(f"Average citations per paper: {AVG_CITATIONS:.2f}")

RESULTADOS
Total papers: 10345
Total citations: 150038
Average citations per paper: 14.50


In [ ]:
# Papers por año
papers_per_year = (
    df.groupby("year")
      .size()
      .reset_index(name="papers")
)

print(papers_per_year)

# Citaciones por año
citations_per_year = (
    df.groupby("year")["citations"]
      .sum()
      .reset_index()
)

print(citations_per_year)


   year  papers
0  2019    1216
1  2020    1404
2  2021    1460
3  2022    1207
4  2023    1497
5  2024    1577
6  2025    1459
7  2026     525
   year  citations
0  2019      37412
1  2020      39368
2  2021      28099
3  2022      20100
4  2023      14414
5  2024       7977
6  2025       2604
7  2026         64


In [ ]:
# Exportar Dataset completo

df.to_csv("engineering_qs_openalex_works.csv", index=False)

# Papers por año
papers_per_year.to_csv("papers_per_year.csv", index=False)

# Citaciones por año
citations_per_year.to_csv("citations_per_year.csv", index=False)

print("Archivos exportados correctamente")

Archivos exportados correctamente


# Nota Importante sobre QS Engineering & Technology

QS utiliza categorías propias, mientras que OpenAlex usa:

* Domains
* Fields
* Subfields
* Topics

La aproximación más correcta en OpenAlex para QS Engineering & Technology es:

> `primary_topic.domain.id:3`



Esto incluye áreas como:

1. Engineering
2. Computer Science
3. Materials Science
4. Electrical Engineering
5. Mechanical Engineering
6. Civil Engineering
7. Artificial Intelligence
8. Robotics
9. Telecommunications